# Module 3: Modifying Data with DDL and DML

**ALY 6420 | Table Structure, Data Types, Constraints, INSERT, UPDATE, DELETE, ALTER, and DROP**

*Course Lecture Notes*


## Module 3

### From reading tables to changing them safely

In the first two modules, the database mostly behaved as something you **read**. This module adds a new responsibility: understanding how tables are built and how data is changed.

The central distinction is simple:

- **DDL (Data Definition Language)** defines or changes database structure.
- **DML (Data Manipulation Language)** reads or changes the rows stored inside that structure.

For analysts, this distinction matters even when most daily work uses `SELECT`. A query result is only as trustworthy as the schema and data behind it.


## Learning objectives

By the end of this lecture, you should be able to:

- distinguish DDL from DML and classify common SQL statements
- select appropriate PostgreSQL data types for common analytical fields
- explain how constraints protect data quality
- write a `CREATE TABLE` statement with useful constraints
- insert one or more rows with `INSERT`
- modify existing rows safely with `UPDATE` and `WHERE`
- remove selected rows safely with `DELETE`
- change table structure with `ALTER TABLE`
- distinguish `DELETE`, `TRUNCATE`, and `DROP`
- use transactions and preview queries to reduce the risk of destructive changes


## Required preparation

Use these resources alongside this lecture:

### Required textbook sections
- Shan et al. (2025), *SQL for Data Analytics* (4th ed.), **Chapter 2: Creating Tables with Solid Structures**
- Shan et al. (2025), **Chapter 6: Transforming and Updating Data**, focusing on updating/deleting data and changing table definitions

### Course environment
- PostgreSQL
- DBeaver
- Pagila database

> **Practice rule:** Run write operations against a practice table or a copy. Do not experiment with destructive statements against the original Pagila tables.


# Part 1: Structure and Data


## Two layers of a relational table

A table has two layers that are easy to confuse:

```text
TABLE
├── Structure (schema)
│   ├── column names
│   ├── data types
│   ├── constraints
│   └── relationships
└── Data
    ├── row 1
    ├── row 2
    └── ...
```

DDL changes the **structure**. DML works with the **rows**.


## DDL and DML at a glance

| Category | Purpose | Common statements in this module |
|---|---|---|
| DDL | Define or change structure | `CREATE`, `ALTER`, `DROP` |
| DML | Read or modify rows | `SELECT`, `INSERT`, `UPDATE`, `DELETE` |

A useful question is: **Am I changing the container, or the contents of the container?**


## CRUD as a data lifecycle

The textbook frames common database work as **CRUD**:

```text
CREATE  →  READ  →  UPDATE  →  DELETE
   ↑                         │
   └──── data lifecycle ─────┘
```

In SQL, these ideas map to several statements. `CREATE TABLE` creates structure, `SELECT` reads data, `UPDATE` changes existing rows, and `DELETE` removes rows. `DROP TABLE` goes further by removing the table itself.


## Quick check: DDL or DML?

Classify each statement before revealing the answer mentally:

1. `ALTER TABLE customer ADD COLUMN loyalty_level VARCHAR(20);`
2. `UPDATE customer SET active = FALSE WHERE customer_id = 17;`
3. `DROP TABLE staging_import;`
4. `INSERT INTO category (name) VALUES ('Documentary');`

**Answers:** 1 and 3 are DDL. 2 and 4 are DML.


# Part 2: Reading a Schema Before Querying


## A table definition is documentation

A `CREATE TABLE` statement answers questions that a sample of rows cannot reliably answer:

- Which columns are required?
- Which values must be unique?
- Which data types are stored?
- Which defaults are supplied automatically?
- Which values are restricted?
- Which columns connect this table to another table?

This is why reading DDL is an analytical skill, not only a database administration skill.


## Example: a compact table definition

```sql
CREATE TABLE support_ticket (
    ticket_id    INT GENERATED ALWAYS AS IDENTITY,
    customer_id  INT NOT NULL,
    subject      VARCHAR(200) NOT NULL,
    status       VARCHAR(20) NOT NULL DEFAULT 'OPEN',
    created_date DATE NOT NULL DEFAULT CURRENT_DATE,
    description  TEXT,
    CONSTRAINT pk_support_ticket
        PRIMARY KEY (ticket_id),
    CONSTRAINT fk_ticket_customer
        FOREIGN KEY (customer_id) REFERENCES customer(customer_id),
    CONSTRAINT chk_ticket_status
        CHECK (status IN ('OPEN', 'IN_PROGRESS', 'CLOSED'))
);
```

Before a single row exists, this schema already tells us what valid data should look like.


## Read the definition like an analyst

From the `support_ticket` definition, you can infer:

- `ticket_id` identifies a row and is generated automatically.
- every ticket must have a `customer_id`, `subject`, `status`, and `created_date`.
- `description` is optional because it has no `NOT NULL` constraint.
- a missing status becomes `OPEN`.
- status values outside the approved list are rejected.
- a customer ID must already exist in `customer`.

These rules directly affect what queries can safely assume.


## Viewing DDL in DBeaver

In DBeaver, inspect an existing Pagila table through its table properties or **View DDL** option.

Try this with:

- `film`
- `customer`
- `rental`
- `payment`

Do not only scan the column names. Look for data types, `NOT NULL`, primary keys, foreign keys, and defaults.


# Part 3: Choosing Data Types


## Data types are analytical decisions

A data type controls both **what can be stored** and **what operations make sense**.

If a date is stored as text, date arithmetic and chronological sorting become unreliable. If money is stored as an approximate floating-point number, rounding can accumulate. A good schema prevents these problems before they reach an analysis.


## Common PostgreSQL types

| Family | Type | Typical use |
|---|---|---|
| Text | `VARCHAR(n)` | names, codes, bounded strings |
| Text | `TEXT` | notes and descriptions |
| Integer | `SMALLINT`, `INT`, `BIGINT` | counts, identifiers, whole numbers |
| Exact numeric | `NUMERIC(p,s)` / `DECIMAL(p,s)` | currency and exact decimals |
| Approximate numeric | `REAL`, `FLOAT` | scientific measurements where approximation is acceptable |
| Date/time | `DATE` | calendar dates |
| Date/time | `TIMESTAMP` | date plus time |
| Date/time | `TIMESTAMPTZ` | date/time values spanning time zones |
| Logical | `BOOLEAN` | true/false flags |


## Example: money should be exact

Suppose an invoice amount must preserve cents.

```sql
amount DECIMAL(10,2)
```

This is preferable to `FLOAT` because financial values normally require exact decimal representation. An `INT` would also be a poor fit if cents must be stored.


## Example: dates should behave like dates

This design invites inconsistent values:

```sql
order_date VARCHAR(50)
```

Possible entries might include `2026-09-05`, `09/05/2026`, and `September 5, 2026`.

A stronger definition is:

```sql
order_date DATE
```

Now PostgreSQL can validate the value and perform date-aware filtering, sorting, and arithmetic.


## A schema diagnosis habit

When a filter produces a surprising result, inspect the type before rewriting the query.

For example:

```sql
WHERE event_date BETWEEN '2026-09-01' AND '2026-09-30'
```

works as intended when `event_date` is a date-compatible type. If it is arbitrary text, the stored representation may undermine the logic.


# Part 4: Constraints as Data Quality Rules


## What is a constraint?

A constraint is a rule enforced by the database when data is written. Instead of hoping every application sends valid data, the table itself rejects values that violate its rules.

Constraints therefore move some data-quality protection closer to the data.


## Six constraints to recognize

| Constraint | Protects against |
|---|---|
| `NOT NULL` | missing required values |
| `DEFAULT` | omitted values that should receive a standard value |
| `UNIQUE` | duplicate values where uniqueness is required |
| `PRIMARY KEY` | rows without a stable unique identifier |
| `FOREIGN KEY` | references to nonexistent parent rows |
| `CHECK` | values outside an allowed condition or range |


## `NOT NULL`

```sql
subject VARCHAR(200) NOT NULL
```

This says that every stored ticket must have a subject.

Without the constraint, the database would permit a row with no subject unless another layer prevented it.


## `DEFAULT`

```sql
status VARCHAR(20) NOT NULL DEFAULT 'OPEN'
```

If an insert omits `status`, PostgreSQL supplies `OPEN`.

A default is not the same as allowing anything. It provides a value when one is omitted.


## `UNIQUE`

```sql
email VARCHAR(255) UNIQUE
```

Use `UNIQUE` when duplicate values would represent a data-quality problem.

Real-world example: if an account system defines one account per email address, a uniqueness rule can stop a second row from silently reusing the same email.


## `PRIMARY KEY`

A primary key uniquely identifies each row.

```sql
CONSTRAINT pk_ticket PRIMARY KEY (ticket_id)
```

A strong identifier makes targeted updates and deletes much safer:

```sql
WHERE ticket_id = 417
```

is usually more precise than filtering by a non-unique description.


## `FOREIGN KEY`

```sql
CONSTRAINT fk_ticket_customer
    FOREIGN KEY (customer_id)
    REFERENCES customer(customer_id)
```

This prevents a ticket from referencing a customer that does not exist.

Foreign keys encode relationships and protect **referential integrity**.


## `CHECK`

```sql
CONSTRAINT chk_ticket_status
    CHECK (status IN ('OPEN', 'IN_PROGRESS', 'CLOSED'))
```

This rejects values such as `DONE`, `Closed`, or `waiting` unless the schema explicitly allows them.

For analytics, consistent categories reduce cleanup later.


## Inline versus table-level constraints

Inline constraints sit beside a column:

```sql
subject VARCHAR(200) NOT NULL
```

Named table-level constraints are useful when you want a clear identifier or need multiple columns:

```sql
CONSTRAINT uq_employee_period
    UNIQUE (employee_id, reporting_week)
```

Meaningful constraint names make later maintenance easier.


## Try it: design the rule

A payroll adjustment table needs these rules:

- each row has a unique generated ID
- employee ID is required
- adjustment amount is exact to cents
- status must be `PENDING`, `APPROVED`, or `REJECTED`
- created date defaults to today

Before continuing, decide which data type and constraint belongs to each requirement.


# Part 5: CREATE TABLE


## The explicit `CREATE TABLE` pattern

```sql
CREATE TABLE table_name (
    column_1 data_type constraints,
    column_2 data_type constraints,
    ...
);
```

Creating a table is not merely naming columns. You are defining the rules future data must follow.


## Worked example: payroll adjustment

```sql
CREATE TABLE payroll_adjustment (
    adjustment_id INT GENERATED ALWAYS AS IDENTITY,
    employee_id   INT NOT NULL,
    amount        DECIMAL(10,2) NOT NULL,
    status        VARCHAR(10) NOT NULL DEFAULT 'PENDING',
    created_date  DATE NOT NULL DEFAULT CURRENT_DATE,
    notes         TEXT,
    CONSTRAINT pk_payroll_adjustment
        PRIMARY KEY (adjustment_id),
    CONSTRAINT chk_adjustment_status
        CHECK (status IN ('PENDING', 'APPROVED', 'REJECTED'))
);
```


## `CREATE TABLE AS SELECT`

Sometimes you need a working copy or derived table based on existing data:

```sql
CREATE TABLE recent_payments AS
SELECT payment_id, customer_id, amount, payment_date
FROM payment
WHERE payment_date >= '2005-08-01';
```

This pattern creates a table from query output. It is useful for practice and staging, but do not assume that every constraint, key, or index from the source table is automatically reproduced.


## Practice safety: copy before changing

For exercises involving `UPDATE`, `DELETE`, `ALTER`, or `DROP`, create a disposable table first.

```sql
CREATE TABLE film_practice AS
SELECT *
FROM film;
```

Then run destructive examples against `film_practice`, not `film`. This keeps the course dataset stable.


# Part 6: INSERT


## `INSERT` adds rows

The safest beginner pattern names the destination columns explicitly:

```sql
INSERT INTO table_name (column_1, column_2, column_3)
VALUES (value_1, value_2, value_3);
```

The number and order of listed values must correspond to the listed columns.


## Single-row insert

```sql
INSERT INTO payroll_adjustment
    (employee_id, amount, notes)
VALUES
    (204, 125.50, 'Travel reimbursement correction');
```

Because `status` and `created_date` have defaults, they can be omitted here.


## Multiple-row insert

```sql
INSERT INTO payroll_adjustment
    (employee_id, amount, status)
VALUES
    (205, 80.00, 'PENDING'),
    (206, 42.75, 'APPROVED'),
    (207, 19.25, 'REJECTED');
```

A single statement can insert multiple rows by separating value lists with commas.


## `INSERT ... SELECT`

Rows can also come from a query:

```sql
INSERT INTO archived_adjustments
    (adjustment_id, employee_id, amount, status, created_date)
SELECT adjustment_id, employee_id, amount, status, created_date
FROM payroll_adjustment
WHERE created_date < '2026-01-01';
```

This pattern is common in staging, archiving, and data-loading workflows.


## Why name columns explicitly?

Compare:

```sql
INSERT INTO payroll_adjustment
VALUES (...);
```

with:

```sql
INSERT INTO payroll_adjustment (employee_id, amount, notes)
VALUES (204, 125.50, 'Correction');
```

The second form communicates intent and is less dependent on the physical column order of the table.


# Part 7: UPDATE


## `UPDATE` changes existing rows

General pattern:

```sql
UPDATE table_name
SET column_name = new_value
WHERE condition;
```

The `WHERE` clause determines **which rows change**. This is the most important safety idea in this part of the module.


## Example: update one row by primary key

```sql
UPDATE payroll_adjustment
SET status = 'APPROVED'
WHERE adjustment_id = 17;
```

A primary-key condition is often the clearest way to target exactly one row.


## Example: update several rows intentionally

```sql
UPDATE film_practice
SET rental_rate = 0.99
WHERE rating = 'G'
  AND rental_duration <= 3;
```

Here, multiple rows may change, but the condition expresses the intended group.


## Updates can use expressions

The new value does not have to be a constant:

```sql
UPDATE product_practice
SET base_msrp = base_msrp * 1.10
WHERE model = 'Model Chi'
  AND year = 2025;
```

The expression uses the current value to calculate the replacement value.


## The dangerous version

```sql
UPDATE film_practice
SET rental_rate = 0.99;
```

There is no `WHERE` clause. Therefore **every row** receives the new rental rate.

SQL does exactly what the statement requests. The database cannot infer that you intended only G-rated films.


## The preview-first pattern

Before an `UPDATE`, run a `SELECT` with the same condition:

```sql
SELECT film_id, title, rental_rate
FROM film_practice
WHERE rating = 'G'
  AND rental_duration <= 3;
```

Check the rows. If they are the intended target, then use the same `WHERE` condition in the `UPDATE`.


## A professional safety question

Before pressing Execute on an `UPDATE`, ask:

> If this condition matches 10,000 rows instead of 10 rows, what happens?

That question encourages you to preview row counts and verify the filter before writing changes.


# Part 8: DELETE, TRUNCATE, and DROP


## `DELETE` removes rows, not the table

```sql
DELETE FROM table_name
WHERE condition;
```

Example:

```sql
DELETE FROM payroll_adjustment
WHERE adjustment_id = 17;
```

Afterward, the table still exists. Only the matching row is removed.


## `DELETE` without `WHERE`

```sql
DELETE FROM payroll_adjustment;
```

This removes **all rows** while leaving the table structure in place.

That can be intentional, but it should never happen because a filter was forgotten.


## `TRUNCATE` clears a table

```sql
TRUNCATE TABLE payroll_adjustment;
```

`TRUNCATE` is designed to remove all rows from a table. It does not accept a row-level `WHERE` filter.

Think of it as an explicit whole-table clearing operation, not a substitute for a targeted `DELETE`.


## `DROP TABLE` removes the object

```sql
DROP TABLE payroll_adjustment;
```

After a successful drop, the table itself no longer exists.

A defensive cleanup pattern is:

```sql
DROP TABLE IF EXISTS payroll_adjustment;
```


## Compare the three

| Statement | Removes selected rows? | Removes all rows? | Keeps table structure? |
|---|---:|---:|---:|
| `DELETE ... WHERE` | Yes | Possibly | Yes |
| `DELETE` without `WHERE` | No filtering | Yes | Yes |
| `TRUNCATE TABLE` | No | Yes | Yes |
| `DROP TABLE` | Not applicable | Yes | **No** |


## Real-world example: retention cleanup

Suppose a staging table contains temporary import records.

- Remove failed rows only: use `DELETE ... WHERE`.
- Empty the staging table before a fresh full reload: `TRUNCATE` may fit.
- Remove an obsolete staging table that should no longer exist: use `DROP TABLE`.

The correct statement depends on whether you are removing **rows** or the **database object**.


# Part 9: ALTER TABLE


## `ALTER TABLE` changes structure after creation

A schema is not frozen forever. Business requirements change. New attributes appear, names improve, and constraints evolve.

`ALTER TABLE` changes an existing table definition without requiring you to recreate the entire table manually.


## Add a column

```sql
ALTER TABLE payroll_adjustment
ADD COLUMN reviewed_by VARCHAR(100);
```

Existing rows have no historical reviewer value, so the new column will normally be `NULL` unless a default or another migration strategy supplies values.


## Add a column with a default

```sql
ALTER TABLE payroll_adjustment
ADD COLUMN active BOOLEAN NOT NULL DEFAULT TRUE;
```

The combination of `NOT NULL` and `DEFAULT` defines both a requirement and a value to use when one is not supplied.


## Rename a column

```sql
ALTER TABLE payroll_adjustment
RENAME COLUMN notes TO adjustment_notes;
```

Renaming may improve clarity, but downstream queries that reference the old name must also be updated.


## Change a column property

```sql
ALTER TABLE payroll_adjustment
ALTER COLUMN status SET NOT NULL;
```

This can fail if existing rows already contain `NULL`. Schema changes must be compatible with the data that already exists.


## Drop a column

```sql
ALTER TABLE payroll_adjustment
DROP COLUMN reviewed_by;
```

Dropping a column removes its stored values and can break queries, reports, or applications that still reference it.


## Schema changes have downstream effects

A database schema is often part of a larger system.

```text
TABLE CHANGE
    ↓
queries
    ↓
dashboards
    ↓
reports
    ↓
applications
```

A technically valid `ALTER TABLE` can still create operational problems if dependent code is not considered.


# Part 10: Transactions and Safe Write Habits


## A transaction groups related work

PostgreSQL lets you wrap changes in a transaction:

```sql
BEGIN;

UPDATE film_practice
SET rental_rate = 0.99
WHERE rating = 'G';

-- inspect the result

COMMIT;
```

`COMMIT` makes the transaction's changes permanent.


## `ROLLBACK` provides an exit before commit

```sql
BEGIN;

UPDATE film_practice
SET rental_rate = 0.99
WHERE rating = 'G';

SELECT title, rental_rate
FROM film_practice
WHERE rating = 'G'
LIMIT 10;

ROLLBACK;
```

`ROLLBACK` undoes the transaction's uncommitted changes.


## PostgreSQL and transactional DDL

PostgreSQL supports transactional behavior for many DDL operations. For example, a table created or dropped inside an explicit transaction can often be rolled back.

Even so, build the habit of treating schema changes as consequential operations. Production tools, permissions, dependencies, and cross-database environments may behave differently.


## A four-step write-safety routine

Before changing data:

1. **Copy or isolate** practice data when appropriate.
2. **Preview** the target rows with `SELECT`.
3. **Execute** the write using the verified condition.
4. **Verify** the result before committing or moving on.

This routine is more valuable than memorizing syntax alone.


## Example: preview, update, verify

```sql
-- 1. Preview
SELECT film_id, title, rental_rate
FROM film_practice
WHERE rating = 'G';

-- 2. Update
UPDATE film_practice
SET rental_rate = 0.99
WHERE rating = 'G';

-- 3. Verify
SELECT film_id, title, rental_rate
FROM film_practice
WHERE rating = 'G';
```


# Part 11: Reading Pagila Schemas


## Pagila example: `payment`

Consider this simplified structure:

```sql
CREATE TABLE payment (
    payment_id   INT GENERATED ALWAYS AS IDENTITY,
    customer_id  SMALLINT NOT NULL,
    staff_id     SMALLINT NOT NULL,
    rental_id    INT,
    amount       DECIMAL(5,2) NOT NULL,
    payment_date TIMESTAMP NOT NULL,
    CONSTRAINT pk_payment PRIMARY KEY (payment_id),
    CONSTRAINT fk_payment_customer
        FOREIGN KEY (customer_id) REFERENCES customer(customer_id)
);
```

The schema provides information you cannot infer safely from a few sample rows.


## What can you infer?

From the definition:

- `payment_id` identifies a payment.
- `customer_id`, `staff_id`, `amount`, and `payment_date` are required.
- `rental_id` may be `NULL`.
- `amount` is stored as an exact decimal.
- `customer_id` must reference an existing customer.

These facts should influence filters, joins, and data-quality checks.


## NULL and filters

Suppose a nullable date column represents an unfinished event.

```sql
WHERE return_date < '2005-07-01'
```

Rows where `return_date` is `NULL` do not satisfy that comparison. They are not automatically “before” or “after” the date.

Schema knowledge helps you recognize when missing values need explicit handling.


# Part 12: Common Failure Modes


## Failure mode 1: wrong data type

**Problem:** storing dates in `VARCHAR`.

**Consequence:** inconsistent formats and unreliable chronological operations.

**Better design:** `DATE`, `TIMESTAMP`, or `TIMESTAMPTZ` according to the requirement.


## Failure mode 2: approximate money

**Problem:** using floating-point storage for exact financial amounts.

**Consequence:** representation and rounding differences can accumulate.

**Better design:** `NUMERIC` or `DECIMAL` with appropriate precision and scale.


## Failure mode 3: no stable identifier

**Problem:** a table has `name`, `date`, and `amount`, but no primary key.

**Consequence:** two rows may look identical, and targeted updates or deletes become harder to express safely.

**Better design:** define a key that uniquely identifies each row.


## Failure mode 4: uncontrolled categories

**Problem:** status is free text.

Possible values become:

```text
Open
OPEN
open
In progress
IN_PROGRESS
Done
Closed
```

**Better design:** when the business domain is truly limited, enforce an agreed set with a `CHECK` constraint or a related lookup table.


## Failure mode 5: missing `WHERE`

```sql
UPDATE employee
SET status = 'INACTIVE';
```

If the intention was one employee, this is catastrophic because every row is targeted.

The safest habit is simple: **write and run the `SELECT` version of the condition first.**


# Part 13: Guided Practice


## Practice A: inspect before writing

Given:

```sql
CREATE TABLE report_data (
    id    INT,
    name  VARCHAR(100),
    val1  FLOAT,
    val2  FLOAT,
    date1 VARCHAR(50),
    flag  VARCHAR(10)
);
```

Before writing queries, identify at least three questions you would ask about the intended meaning of these columns.


## Practice A: possible concerns

Questions might include:

- Is `id` supposed to uniquely identify a row?
- What do `val1` and `val2` measure? Are they currency, counts, percentages, or measurements?
- Is `date1` truly a date? What formats can appear?
- What values are valid for `flag`?
- Which columns are required?

Notice that ambiguity is itself a schema-quality problem.


## Practice B: strengthen the schema

Suppose you learn that:

- `id` is a unique required identifier
- `val1` is a monetary amount
- `date1` is a required calendar date
- `flag` must be `Y` or `N`

A stronger design could use:

```sql
id    INT PRIMARY KEY,
val1  DECIMAL(10,2),
date1 DATE NOT NULL,
flag  CHAR(1) CHECK (flag IN ('Y', 'N'))
```


## Practice C: predict the effect

What does this statement do?

```sql
UPDATE report_data
SET flag = 'N';
```

**Answer:** it changes `flag` to `N` for every row in the table because no `WHERE` clause limits the target.


## Practice D: make it safer

Suppose only rows before January 1, 2025 should change. Preview first:

```sql
SELECT id, date1, flag
FROM report_data
WHERE date1 < '2025-01-01';
```

Then, if the result is correct:

```sql
UPDATE report_data
SET flag = 'N'
WHERE date1 < '2025-01-01';
```


# Part 14: Integrated Worked Example


## Scenario: service request tracking

A company needs a small table for service requests. Requirements:

- generated request ID
- required customer ID
- required short subject
- status limited to three values
- created date defaults to today
- optional description

We will move through the lifecycle: create, insert, update, alter, delete, and clean up.


## Step 1: create

```sql
CREATE TABLE service_request (
    request_id   INT GENERATED ALWAYS AS IDENTITY,
    customer_id  INT NOT NULL,
    subject      VARCHAR(200) NOT NULL,
    status       VARCHAR(20) NOT NULL DEFAULT 'OPEN',
    created_date DATE NOT NULL DEFAULT CURRENT_DATE,
    description  TEXT,
    CONSTRAINT pk_service_request PRIMARY KEY (request_id),
    CONSTRAINT chk_service_request_status
        CHECK (status IN ('OPEN', 'IN_PROGRESS', 'CLOSED'))
);
```


## Step 2: insert

```sql
INSERT INTO service_request
    (customer_id, subject, description)
VALUES
    (101, 'Incorrect invoice', 'Customer reports duplicate charge.'),
    (205, 'Address change', 'Update mailing address.');
```

The omitted status and date columns receive their defaults.


## Step 3: inspect

```sql
SELECT request_id, customer_id, subject, status, created_date
FROM service_request
ORDER BY request_id;
```

Always inspect what exists before deciding what to change.


## Step 4: update one request

First preview:

```sql
SELECT *
FROM service_request
WHERE request_id = 1;
```

Then update:

```sql
UPDATE service_request
SET status = 'IN_PROGRESS'
WHERE request_id = 1;
```


## Step 5: evolve the schema

A new requirement asks for the person who resolved a request:

```sql
ALTER TABLE service_request
ADD COLUMN resolved_by VARCHAR(100);
```

The table can evolve as the business process evolves.


## Step 6: delete a mistaken row

Preview:

```sql
SELECT *
FROM service_request
WHERE request_id = 2;
```

Delete only after verifying:

```sql
DELETE FROM service_request
WHERE request_id = 2;
```


## Step 7: clean up the practice object

```sql
DROP TABLE IF EXISTS service_request;
```

This final step removes both the practice data and its table definition.


# Part 15: What to Remember


## Decision map

```text
Need a new table?              → CREATE TABLE
Need a new row?                → INSERT
Need to read rows?             → SELECT
Need to change existing rows?  → UPDATE ... WHERE
Need to remove selected rows?  → DELETE ... WHERE
Need to empty a table?         → TRUNCATE or intentional DELETE
Need to change table structure?→ ALTER TABLE
Need to remove the table?      → DROP TABLE
```


## Five habits that prevent many SQL mistakes

1. **Read the schema before assuming what a column means.**
2. **Use data types that match the real-world value.**
3. **Use constraints to prevent invalid data at write time.**
4. **Preview `UPDATE` and `DELETE` targets with `SELECT`.**
5. **Practice destructive operations on copies, not source tables.**


## Knowledge check

Answer without running SQL first:

1. What is the difference between DDL and DML?
2. Why is `DECIMAL(10,2)` usually preferable to `FLOAT` for money?
3. What does a foreign key protect?
4. What happens if `UPDATE` has no `WHERE` clause?
5. How does `DELETE` differ from `DROP TABLE`?
6. Why might `ALTER COLUMN ... SET NOT NULL` fail on an existing table?
7. What should you do before running a destructive write statement?


## Knowledge check answers

1. DDL defines or changes structure; DML reads or changes stored rows.
2. `DECIMAL` provides exact decimal representation appropriate for financial values.
3. A foreign key prevents references to nonexistent parent keys and supports referential integrity.
4. Every row in the table is updated.
5. `DELETE` removes rows but keeps the table; `DROP TABLE` removes the table definition and its data.
6. Existing rows may contain `NULL`, which would violate the new constraint.
7. Preview the target rows with a `SELECT`, and use a practice copy or transaction when appropriate.


## Module 3 summary

This module moves from querying existing tables to understanding and safely changing the structures and rows behind them.

The key ideas are:

- schemas encode meaning through types, constraints, and relationships
- DDL changes structure; DML works with data
- `CREATE TABLE` should express data-quality rules, not only column names
- `INSERT` adds rows
- `UPDATE` and `DELETE` require deliberate targeting
- `ALTER TABLE` changes an existing schema
- `DELETE`, `TRUNCATE`, and `DROP` remove different things
- previewing, verifying, and using transactions are core safety habits

The next module uses the relationships expressed through keys to combine data across tables with joins.


## References

- Shan, J., Li, H., Goldwasser, M., Malik, U., & Johnston, B. (2025). *SQL for Data Analytics: Analyze Data Effectively, Uncover Insights and Master Advanced SQL for Real-World Applications* (4th ed.). Packt Publishing.
- PostgreSQL Global Development Group. *PostgreSQL 16 Documentation: Data Types and Data Definition*.
- PostgreSQL Exercises. *Simple SQL Queries Exercises*.
- Pagila sample database, used throughout the course for PostgreSQL query practice.
